# Indus Valley AI — Fine-tune Mistral-7B with QLoRA

**Author:** Mohammed Majeed Khan, AI Hub, Mahindra University

This notebook fine-tunes **Mistral-7B-Instruct** on the Indus Valley AI knowledge base using QLoRA (4-bit quantized LoRA). It runs on a free Google Colab T4 GPU in approximately 4–6 hours.

## Pipeline
1. Install dependencies
2. Upload `training_data.jsonl` (generated by `extract_data.py`)
3. Load Mistral-7B base model in 4-bit
4. Apply LoRA adapters
5. Train for 3 epochs
6. Save & test the model
7. (Optional) Push to Hugging Face Hub for free hosting

## Cost
**$0** — uses Google Colab's free T4 GPU. If training is slow, Colab Pro (~$10/month) gives faster GPUs.

## Step 1 — Verify GPU is available

In [ ]:
!nvidia-smi

## Step 2 — Install dependencies
Takes ~2 minutes. Restart runtime if prompted.

In [ ]:
!pip install -q -U bitsandbytes
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install -q -U git+https://github.com/huggingface/accelerate.git
!pip install -q datasets trl scipy sentencepiece

## Step 3 — Upload your training data

Upload `training_data.jsonl` (generated locally by `extract_data.py`) using Colab's file uploader.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload training_data.jsonl

## Step 4 — Load and format the dataset

In [ ]:
import json
from datasets import Dataset

examples = []
with open('training_data.jsonl') as f:
    for line in f:
        examples.append(json.loads(line))
print(f'Loaded {len(examples)} training examples')

def format_for_mistral(ex):
    sys_msg = ex.get('system', '')
    user = ex['instruction']
    assistant = ex['output']
    text = f'<s>[INST] {sys_msg}\n\n{user} [/INST] {assistant}</s>'
    return {'text': text}

ds = Dataset.from_list([format_for_mistral(e) for e in examples])
ds = ds.train_test_split(test_size=0.05, seed=42)
print(ds)

## Step 5 — Load Mistral-7B base model in 4-bit

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.2'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
model.config.pretraining_tp = 1
print('Model loaded.')

## Step 6 — Apply LoRA adapters
Only ~0.2% of parameters become trainable, but it's enough to specialize the model on Indus content.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 7 — Train (3 epochs, ~4–6 hours on T4)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./indus-valley-ai-mistral7b',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim='paged_adamw_32bit',
    save_strategy='epoch',
    logging_steps=10,
    learning_rate=2e-4,
    bf16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds['train'],
    eval_dataset=ds['test'],
    dataset_text_field='text',
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_args,
    packing=False,
)

trainer.train()

## Step 8 — Save the fine-tuned adapter

In [ ]:
trainer.model.save_pretrained('indus-valley-ai-final')
tokenizer.save_pretrained('indus-valley-ai-final')
print('Saved.')

## Step 9 — Test it

In [ ]:
from transformers import pipeline

pipe = pipeline('text-generation', model=trainer.model, tokenizer=tokenizer, max_new_tokens=400)

questions = [
    'What is the unicorn seal?',
    'Tell me about the Great Bath of Mohenjo-daro.',
    'How many wells did Mohenjo-daro have?',
    'What does positional analysis tell us about the Indus script?',
    'Explain the FISH-JAR bigram.'
]

SYS = 'You are Indus Valley AI, a domain-restricted scholarly assistant on the Indus / Harappan civilization (c. 3300\u20131300 BCE).'
for q in questions:
    prompt = f'<s>[INST] {SYS}\n\n{q} [/INST]'
    out = pipe(prompt, do_sample=False)[0]['generated_text']
    print('Q:', q)
    print('A:', out.split('[/INST]')[-1].strip())
    print('---')

## Step 10 — (Optional) Push to Hugging Face Hub for free hosting

In [ ]:
from huggingface_hub import login
login()  # paste your HF token from https://huggingface.co/settings/tokens

trainer.model.push_to_hub('YOUR_HF_USERNAME/indus-valley-ai-mistral7b')
tokenizer.push_to_hub('YOUR_HF_USERNAME/indus-valley-ai-mistral7b')

Once pushed, deploy a free **Hugging Face Inference Endpoint** and call it from your website's `app.js` to get real LLM-generated answers.